# Deterministic PEM hydrogen NPV

Calculate expected-input NPV, levelized net margin, and LCOH at 100,000 tH2/year. The source model uses a 25-year lifetime and 7,500 EUR/tH2 retail price. Uncertain inputs use their analytical means. Greenfield European proton-exchange-membrane electrolysis for 2030, with purchased electricity specified at 200 bar.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from hydrogen.hydrogen_npv_deterministic import calculate_deterministic_hydrogen_result
from hydrogen.hydrogen_npv_summary_figures import (
    HYDROGEN_PROCESSED_OUTPUT_COLUMNS,
    HYDROGEN_RAW_INPUT_COLUMNS,
    HYDROGEN_TECHNOLOGY_LABELS,
)

pd.options.display.float_format = "{:,.3f}".format

In [2]:
TECHNOLOGY = "pem"
result = calculate_deterministic_hydrogen_result(TECHNOLOGY)
values = {key: item[0] for key, item in result.items()}

UNITS = {
    "annual_output_th2": "tH2/year",
    "lifetime_years": "years",
    "capex_eur_per_th2": "EUR/(tH2/y)",
    "fixed_opex_eur_per_th2": "EUR/tH2",
    "variable_opex_eur_per_th2": "EUR/tH2",
    "natural_gas_consumption_mwh_per_th2": "MWh/tH2",
    "biomethane_consumption_mwh_per_th2": "MWh/tH2",
    "biomass_consumption_mwh_per_th2": "MWh/tH2",
    "electricity_consumption_mwh_per_th2": "MWh/tH2",
    "emissions_tco2_per_th2": "tCO2/tH2",
    "gas_price_eur_per_mwh_th": "EUR/MWh_th",
    "biomethane_price_eur_per_mwh_th": "EUR/MWh_th",
    "biomass_price_eur_per_mwh_th": "EUR/MWh_th",
    "electricity_price_eur_per_mwh": "EUR/MWh",
    "hydrogen_price_eur_per_th2": "EUR/tH2",
    "carbon_price_eur_per_t": "EUR/tCO2",
    "transport_and_storage_share_of_capture_cost": "fraction",
    "transport_and_storage_cost_eur_per_th2": "EUR/tH2",
    "capture_cost_excluding_transport_and_storage_eur_per_th2": "EUR/tH2",
    "discounted_lifetime_output_th2": "discounted tH2",
    "lcoh_eur_per_th2": "EUR/tH2",
    "levelized_net_margin_eur_per_th2": "EUR/tH2",
    "capture_fraction": "fraction",
}

def unit_for(key):
    key = key.removeprefix("bau_").replace("_change_", "_")
    if key in UNITS:
        return UNITS[key]
    if key.endswith("_eur"):
        return "EUR"
    if key in {"technology", "technology_type", "retrofit_bau_mode", "fuel_type"}:
        return "category"
    return ""

def as_table(keys):
    return pd.DataFrame(
        {"Input / output": key, "Value": values[key], "Unit": unit_for(key)}
        for key in keys if key in values and key != "run_id"
    )

summary = pd.DataFrame([
    {"Metric": "Technology", "Value": HYDROGEN_TECHNOLOGY_LABELS[TECHNOLOGY], "Unit": ""},
    {"Metric": "Annual hydrogen output", "Value": values["annual_output_th2"], "Unit": "tH2/year"},
    {"Metric": "Direct emissions", "Value": values["emissions_tco2_per_th2"], "Unit": "tCO2/tH2"},
    {"Metric": "NPV", "Value": values["npv_eur"] / 1_000_000, "Unit": "million EUR"},
    {"Metric": "Levelized net margin", "Value": values["levelized_net_margin_eur_per_th2"], "Unit": "EUR/tH2"},
    {"Metric": "LCOH", "Value": values["lcoh_eur_per_th2"], "Unit": "EUR/tH2"},
])
raw_inputs = as_table(HYDROGEN_RAW_INPUT_COLUMNS)
processed_outputs = as_table(HYDROGEN_PROCESSED_OUTPUT_COLUMNS)
retrofit_keys = [
    key for key in values
    if key.startswith("bau_") or "_change_" in key or key == "capture_fraction"
]
if TECHNOLOGY == "biomethane_smr":
    retrofit_keys.append("biomethane_consumption_mwh_per_th2")
retrofit_inputs = as_table(retrofit_keys)

## Summary

In [3]:
summary

,Metric,Value,Unit
0,Technology,PEM,
1,Annual hydrogen output,"100,000.000",tH2/year
2,Direct emissions,0.000,tCO2/tH2
3,NPV,121.899,million EUR
4,Levelized net margin,114.193,EUR/tH2
5,LCOH,"7,385.807",EUR/tH2


## Expected inputs

In [4]:
raw_inputs

,Input / output,Value,Unit
0,technology,pem,category
1,technology_type,absolute,category
2,retrofit_bau_mode,not_applicable,category
3,annual_output_th2,"100,000.000",tH2/year
4,lifetime_years,25,years
5,capex_eur_per_th2,"10,958.000",EUR/(tH2/y)
6,fixed_opex_eur_per_th2,388.333,EUR/tH2
7,variable_opex_eur_per_th2,38.333,EUR/tH2
8,fuel_type,none,category
9,natural_gas_consumption_mwh_per_th2,0.000,MWh/tH2


## Processed outputs

In [5]:
processed_outputs

,Input / output,Value,Unit
0,technology,pem,category
1,technology_type,absolute,category
2,retrofit_bau_mode,not_applicable,category
3,initial_capex_eur,"1,095,800,000.000",EUR
4,annual_revenue_eur,"750,000,000.000",EUR
5,annual_fixed_opex_eur,"38,833,333.333",EUR
6,annual_variable_opex_eur,"3,833,333.333",EUR
7,annual_natural_gas_cost_eur,0.000,EUR
8,annual_biomethane_cost_eur,0.000,EUR
9,annual_biomass_cost_eur,0.000,EUR
